# Trade events to bar-level facts

Aggregate event-level PAXG trades into OHLC, activity, flow, size and sequence facts. This is a PAXG example, not an ETH downloader.

**Status:** curated research source; original saved outputs preserved. Original notes about results are historical observations, not revalidated conclusions. Read `docs/DATA_AND_RUNNING.md` before execution.

Source: `Phase 0.ipynb`, original cells 0–69 (zero-based).


In [ ]:
from pathlib import Path
import os
import sys

# Locate the repository from its root or any notebook subdirectory.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "tradinguni_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the TradingUni repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from tradinguni_paths import data_path, output_path
os.chdir(REPO_ROOT)


In [2]:
# Dependencies are installed once from requirements.txt.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.7/805.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 MB 98.4 MB/s eta 0:00:00


In [1]:
import polars as pl

In [2]:
df = pl.read_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/PAXGUSDTaggTrades/aggTrades-PAXG-2026-01-01-28.parquet")))

In [4]:
df = df.with_columns(
    pl.col("timestamp").cast(pl.Int64).alias("ts_us")
)

In [6]:
cutoff = pl.datetime(2025, 1, 1).cast(pl.Int64)

In [7]:
df = df.with_columns(
    pl.when(pl.col("ts_us") > cutoff)
      .then(pl.col("ts_us") // 1000)   # μs → ms
      .otherwise(pl.col("ts_us"))      # เดิมคือ ms อยู่แล้ว
      .alias("ts_fixed_ms")
)

In [8]:
df = df.with_columns(
    pl.col("ts_fixed_ms")
      .cast(pl.Datetime("ms"))
      .alias("timestamp")
)

In [9]:
df

aggTradeId,price,quantity,firstTradeId,lastTradeId,timestamp,isBuyerMaker,isBestMatch,ts_us,ts_fixed_ms
i64,f64,f64,i64,i64,datetime[ms],bool,bool,i64,i64
22348038,4337.0,0.007,31213149,31213149,2026-01-01 00:00:03.140,false,true,1767225603140582,1767225603140
22348039,4337.0,0.0069,31213150,31213150,2026-01-01 00:00:03.184,false,true,1767225603184429,1767225603184
22348040,4336.99,0.0035,31213151,31213151,2026-01-01 00:00:21.965,true,true,1767225621965003,1767225621965
22348041,4337.0,0.0011,31213152,31213152,2026-01-01 00:00:25.126,false,true,1767225625126211,1767225625126
22348042,4337.0,0.0013,31213153,31213153,2026-01-01 00:00:35.701,false,true,1767225635701958,1767225635701
…,…,…,…,…,…,…,…,…,…
24053161,5541.99,0.001,33480472,33480472,2026-01-28 23:59:58.858,true,true,1769644798858992,1769644798858
24053162,5541.99,0.002,33480473,33480473,2026-01-28 23:59:58.860,true,true,1769644798860902,1769644798860
24053163,5541.99,0.0014,33480474,33480474,2026-01-28 23:59:59.730,true,true,1769644799730766,1769644799730


In [10]:
df = (
    df
    .select(["aggTradeId", "timestamp", "price", "quantity", "isBuyerMaker"])
    .select([
        pl.col("aggTradeId"),
        pl.col("timestamp")
          .dt.truncate("1ms")
          .dt.cast_time_unit("ms")
          .alias("t"),
        pl.col("price").alias("p"),
        pl.col("quantity").alias("q"),
        pl.when(pl.col("isBuyerMaker"))
          .then(pl.lit(-1))
          .otherwise(pl.lit(1))
          .cast(pl.Int8)
          .alias("s"),
    ])
    .unique(subset=["aggTradeId"], keep="first")
)

In [17]:
# import polars as pl

# df = (
#     pl.read_parquet(
#         "/content/drive/MyDrive/LoserToWinner/0.Data -> Facts/Binance Vision Data/PAXGUSDTaggTrades/aggTrades-PAXG-2020-2025.parquet",
#         columns=["aggTradeId", "timestamp", "price", "quantity", "isBuyerMaker"],
#     )
#     .select([
#         pl.col("aggTradeId"),
#         pl.col("timestamp").dt.truncate("1ms").dt.cast_time_unit("ms").alias("t"),
#         pl.col("price").alias("p"),
#         pl.col("quantity").alias("q"),
#         pl.when(pl.col("isBuyerMaker")).then(pl.lit(-1)).otherwise(pl.lit(1))
#           .cast(pl.Int8)
#           .alias("s"),
#     ])
#     .unique(subset=["aggTradeId"], keep="first")
# )

# df

In [20]:
df

aggTradeId,t,p,q,s
i64,datetime[ms],f64,f64,i8
20822046,2025-11-17 16:41:41.801,4058.86,1.8576,-1
21986436,2025-12-22 22:13:11.430,4468.7,0.012,-1
2595780,2021-05-20 10:14:46.843,1873.62,0.032376,-1
12774789,2025-04-22 22:05:35.810,3341.0,0.15,1
22014357,2025-12-23 12:41:38.621,4504.04,0.5878,-1
…,…,…,…,…
18193754,2025-10-17 03:17:41.717,4392.33,0.045,1
10260539,2024-11-07 18:43:18.580,2693.0,0.0187,1
446715,2021-01-08 13:09:49.775,1913.98,0.020657,-1


In [11]:
gaps = (
    df.select("t").sort("t")
    .with_columns([
        pl.col("t").shift(1).alias("t_prev"),
        (pl.col("t") - pl.col("t").shift(1)).alias("dt"),
    ])
    .filter(pl.col("dt") > pl.duration(minutes=1))
)

gaps.sort("dt", descending=True)

t,t_prev,dt
datetime[ms],datetime[ms],duration[ms]
2026-01-17 07:38:49.275,2026-01-17 07:34:01.074,4m 48s 201ms
2026-01-01 03:14:52.259,2026-01-01 03:11:12.088,3m 40s 171ms
2026-01-10 07:36:41.592,2026-01-10 07:33:13.642,3m 27s 950ms
2026-01-11 03:50:11.454,2026-01-11 03:46:56.560,3m 14s 894ms
2026-01-11 07:36:24.376,2026-01-11 07:33:22.438,3m 1s 938ms
…,…,…
2026-01-09 03:37:01.295,2026-01-09 03:36:01.254,1m 41ms
2026-01-17 20:58:15.802,2026-01-17 20:57:15.766,1m 36ms
2026-01-09 03:43:01.258,2026-01-09 03:42:01.234,1m 24ms


Facts ที่ต้องได้ต่อ 1 bar

A: OHLC, vwap, logret, range/hl_pct, realized_var_inbar (จาก event log returns)

B: trade_count, trade_rate, dt_mean/median/p90 (dt ภายใน bar), frac_dt0, active_ms (unique timestamp_ms), intensity_peak (max trades per 1s ภายใน bar)

C: vol_base, vol_quote, avg_trade_q, avg_trade_notional, q_p50/p90/p99, notional_p90/p99, top1_share (ตาม threshold p99)

D: buy/sell count & vol (แนะนำ quote), signed_quote, imbalance

E: sign_persistence_inbar, flip_count, run_max, run_mean, sign_entropy (แบบ sequence-based)

F: dp0_frac, n_price_changes, up_ticks, down_ticks, max_up_move, max_down_move

# Step 1: ทำ AggBar “แกนหลัก” ก่อน (A + ส่วนหนึ่งของ B/C/D)

A: open/high/low/close, vwap, ret, range, hl_pct ✅

B: trade_count, trade_rate ✅

C: vol_base, vol_quote ✅

D: buy_count/sell_count, buy_quote/sell_quote, signed_quote, imbalance ✅

In [12]:
bar_seconds = 60*1

In [13]:
# -------------------------
# QUALITY THRESHOLDS
# -------------------------
MIN_TRADES_OHLC = 1          # OHLC ใช้ได้ตั้งแต่มี trade
MIN_TRADES_STATS = 5         # dt / intensity / dp0_frac เริ่มพอมีความหมาย
MIN_TRADES_ENTROPY = 20      # entropy / persistence / runs meaningful
MIN_TRADES_TOP1 = 50         # top1_share meaningful (p99 ใน sample เล็ก bias มาก)
MIN_RETURNS_RV = 5           # realized_var meaningful ต้องมี returns พอสมควร

In [14]:
# 1) sort ให้ sequence ภายใน bar ถูก
df0 = (
    df
    .sort(["t", "aggTradeId"])
)

In [15]:
# 2) ทำ agg ด้วย group_by_dynamic เป็น 5s window
bars_core = (
    df0
    .group_by_dynamic(
        index_column="t",
        every=f"{bar_seconds}s",
        period=f"{bar_seconds}s",
        closed="left",
        label="left",
        include_boundaries=False,
    )
    .agg([
        # A) OHLC
        pl.col("p").first().alias("open"),
        pl.col("p").max().alias("high"),
        pl.col("p").min().alias("low"),
        pl.col("p").last().alias("close"),

        # C) volume
        pl.col("q").sum().alias("vol_base"),
        (pl.col("p") * pl.col("q")).sum().alias("vol_quote"),

        # B) activity
        pl.len().alias("trade_count"),

        # D) direction
        (pl.col("s") == 1).sum().alias("buy_count"),
        (pl.col("s") == -1).sum().alias("sell_count"),

        # quote vol ตามฝั่ง
        (pl.when(pl.col("s") == 1)
            .then(pl.col("p") * pl.col("q"))
            .otherwise(0.0)
        ).sum().alias("buy_quote"),

        (pl.when(pl.col("s") == -1)
            .then(pl.col("p") * pl.col("q"))
            .otherwise(0.0)
        ).sum().alias("sell_quote"),

        # signed quote = sum(s*p*q)
        (pl.col("s").cast(pl.Float64) * pl.col("p") * pl.col("q")).sum().alias("signed_quote"),
    ])
    .with_columns([
        # vwap
        (pl.col("vol_quote") / pl.col("vol_base")).alias("vwap"),

        # trade_rate
        (pl.col("trade_count") / pl.lit(bar_seconds)).alias("trade_rate"),

        # imbalance = signed_quote / vol_quote  (กันหาร 0)
        pl.when(pl.col("vol_quote") > 0)
          .then(pl.col("signed_quote") / pl.col("vol_quote"))
          .otherwise(None)
          .alias("imbalance"),

        # range
        (pl.col("high") - pl.col("low")).alias("range"),
    ])
    .with_columns([
        # hl_pct
        pl.when(pl.col("vwap") > 0)
          .then(pl.col("range") / pl.col("vwap"))
          .otherwise(None)
          .alias("hl_pct"),

        # logret(open->close)
        pl.when((pl.col("open") > 0) & (pl.col("close") > 0))
          .then((pl.col("close") / pl.col("open")).log())
          .otherwise(None)
          .alias("ret"),
    ])
)

# Step 2: ทำ “ต่อเนื่องระหว่างบาร์” ให้ครบ (prev_close + ret_close_to_close + gap)

prev_close = close ของบาร์ก่อน

ret_cc = log(close / prev_close)

gap_oc = log(open / prev_close) (overnight/gap ระหว่างบาร์)

ret_inbar (rename ของเดิม) เพื่อไม่ให้สับสน

In [16]:
bars_core = (
    bars_core
    .with_columns([
        pl.col("close").shift(1).alias("prev_close"),
    ])
    .with_columns([
        # rename ให้ชัด
        pl.col("ret").alias("ret_inbar"),

        # close-to-close
        pl.when((pl.col("prev_close") > 0) & (pl.col("close") > 0))
          .then((pl.col("close") / pl.col("prev_close")).log())
          .otherwise(None)
          .alias("ret_cc"),

        # gap open vs prev_close
        pl.when((pl.col("prev_close") > 0) & (pl.col("open") > 0))
          .then((pl.col("open") / pl.col("prev_close")).log())
          .otherwise(None)
          .alias("gap_oc"),
    ])
    .drop("ret")  # ถ้าอยากให้เหลือชื่อเดียว
)

# Step 3: B) dt_mean/median/p90 + frac_dt0 + active_ms + intensity_peak

In [17]:
bars_time_stats = (
    df0
    .group_by_dynamic(
        index_column="t",
        every=f"{bar_seconds}s",
        period=f"{bar_seconds}s",
        closed="left",
        label="left",
    )
    .agg([
        # dt series ภายในบาร์
        pl.col("t").diff().dt.total_milliseconds().drop_nulls().mean().alias("dt_mean_ms"),
        pl.col("t").diff().dt.total_milliseconds().drop_nulls().median().alias("dt_median_ms"),
        pl.col("t").diff().dt.total_milliseconds().drop_nulls().quantile(0.90, "nearest").alias("dt_p90_ms"),

        # frac_dt0: สัดส่วน dt==0 (ในบาร์)
        (pl.col("t").diff().dt.total_milliseconds().drop_nulls() == 0).mean().alias("frac_dt0"),

        # active_ms: unique timestamp_ms ในบาร์
        pl.col("t").dt.epoch("ms").n_unique().alias("active_ms"),

        # # dt series ภายในบาร์
        # pl.col("t").diff().dt.total_microseconds()/1000.drop_nulls().mean().alias("dt_mean_ms"),
        # pl.col("t").diff().dt.total_microseconds()/1000.drop_nulls().median().alias("dt_median_ms"),
        # pl.col("t").diff().dt.total_microseconds()/1000.drop_nulls().quantile(0.90, "nearest").alias("dt_p90_ms"),

        # # frac_dt0: สัดส่วน dt==0 (ในบาร์)
        # (pl.col("t").diff().dt.total_microseconds()/1000.drop_nulls() == 0).mean().alias("frac_dt0"),

        # pl.col("t").dt.epoch("us").n_unique()/1000.alias("active_ms")
    ])
)

# intensity_peak: max trades per 1s ภายในบาร์
intensity_peak = (
    df0
    .with_columns([
        pl.col("t").dt.truncate(f"{bar_seconds}s").alias("bar_t"),
        pl.col("t").dt.truncate("1s").alias("sec_t"),
    ])
    .group_by(["bar_t", "sec_t"])
    .agg(pl.len().alias("trades_1s"))
    .group_by("bar_t")
    .agg(pl.col("trades_1s").max().alias("intensity_peak_1s"))
    .rename({"bar_t": "t"})  # ให้ join กับ bars_core ได้
)

bars_core = (
    bars_core
    .join(bars_time_stats, on="t", how="left")
    .join(intensity_peak, on="t", how="left")
)

In [18]:
del bars_time_stats, intensity_peak
import gc
gc.collect()

10

# Step 4: A + F จาก event-level ภายใน 5s bar

A) realized_var_inbar

- realized_var_inbar = sum( (Δlogp_event)^2 ) ภายในบาร์

F) price-change micro proxies

- dp0_frac = mean(Δp == 0)

- n_price_changes = count(Δp != 0)

- up_ticks = count(Δp > 0)

- down_ticks = count(Δp < 0)

- max_up_move = max(Δp)

- max_down_move = min(Δp) (จะเป็นค่าติดลบ ถ้าอยากเป็น magnitude ค่อยทำ abs ทีหลัง)

หมายเหตุ: ถ้าในบาร์มี trade เดียว → ไม่มี diff → metric พวกนี้จะเป็น null (นิยามตรงไปตรงมา)

In [19]:
bars_price = (
    df0
    .group_by_dynamic(
        index_column="t",
        every=f"{bar_seconds}s",
        period=f"{bar_seconds}s",
        closed="left",
        label="left",
    )
    .agg([
        # --- A) realized variance in bar: sum( (Δlogp)^2 ) ---
        (
            pl.col("p")
            .log()
            .diff()
            .drop_nulls()
            .pow(2)
            .sum()
        ).alias("realized_var_inbar"),

        # --- F) dp series ---
        (pl.col("p").diff().drop_nulls() == 0).mean().alias("dp0_frac"),
        (pl.col("p").diff().drop_nulls() != 0).sum().alias("n_price_changes"),

        (pl.col("p").diff().drop_nulls() > 0).sum().alias("up_ticks"),
        (pl.col("p").diff().drop_nulls() < 0).sum().alias("down_ticks"),

        pl.col("p").diff().drop_nulls().max().alias("max_up_move"),
        pl.col("p").diff().drop_nulls().min().alias("max_down_move"),
    ])
)

bars_core = bars_core.join(bars_price, on="t", how="left")

In [20]:
bars_core = bars_core.with_columns([
    pl.when(pl.col("trade_count") > 1)
      .then(pl.col("trade_count") - 1)
      .otherwise(0)
      .alias("n_returns"),
])

In [21]:
del bars_price
gc.collect()

0

# Step 5: Size / Notional Quantiles + Concentration

จาก event-level `q` และ `notional = p*q`

**C1) basic averages**

* `avg_trade_q`
* `avg_trade_notional`

**C2) quantiles (tail shape)**

* `q_p50`, `q_p90`, `q_p99`
* `notional_p90`, `notional_p99`

**C3) concentration**

* `top1_share`

  * นิยาม: sum(q | q>=p99)/sum(q)

  * (ถ้าคุณอยากใช้ notional ก็เปลี่ยนเป็น `p*q` ได้ทันที)

In [22]:
bars_size = (
    df0
    .with_columns([
        (pl.col("p") * pl.col("q")).alias("notional"),
    ])
    .group_by_dynamic(
        index_column="t",
        every=f"{bar_seconds}s",
        period=f"{bar_seconds}s",
        closed="left",
        label="left",
    )
    .agg([
        # --- averages ---
        pl.col("q").mean().alias("avg_trade_q"),
        pl.col("notional").mean().alias("avg_trade_notional"),

        # --- quantiles of q ---
        pl.col("q").quantile(0.50, "nearest").alias("q_p50"),
        pl.col("q").quantile(0.90, "nearest").alias("q_p90"),
        pl.col("q").quantile(0.99, "nearest").alias("q_p99"),

        # --- quantiles of notional ---
        pl.col("notional").quantile(0.90, "nearest").alias("notional_p90"),
        pl.col("notional").quantile(0.99, "nearest").alias("notional_p99"),

        # total volume (ไว้ใช้คำนวณ share)
        pl.col("q").sum().alias("_vol_base_tmp"),
    ])
)

คำนวณ top1_share

In [23]:
# 0) สร้าง key บาร์ ให้ event ทุกแถว
df_bar = df0.with_columns(
    pl.col("t").dt.truncate(f"{bar_seconds}s").alias("bar_t")
)

# 1) ต่อบาร์: q_p99 + vol_base
bar_tail = (
    df_bar
    .group_by("bar_t")
    .agg([
        pl.col("q").quantile(0.99, "nearest").alias("q_p99"),
        pl.col("q").sum().alias("vol_base"),
    ])
)

# 2) sum(q | q >= q_p99) ต่อบาร์
top1_sum = (
    df_bar
    .join(bar_tail, on="bar_t", how="left")
    .filter(pl.col("q") >= pl.col("q_p99"))
    .group_by("bar_t")
    .agg(pl.col("q").sum().alias("top1_q"))
)

# 3) top1_share = top1_q / vol_base
top1_share = (
    bar_tail
    .join(top1_sum, on="bar_t", how="left")
    .with_columns([
        pl.col("top1_q").fill_null(0.0),
        pl.when(pl.col("vol_base") > 0)
          .then(pl.col("top1_q") / pl.col("vol_base"))
          .otherwise(None)
          .alias("top1_share")
    ])
    .select([
        pl.col("bar_t").alias("t"),
        pl.col("top1_share")
    ])
)

In [24]:
bars_core = (
    bars_core
    .join(bars_size.drop("_vol_base_tmp"), on="t", how="left")
    .join(top1_share, on="t", how="left")
)

In [25]:
bars_core = bars_core.with_columns([
    pl.when(pl.col("trade_count") >= MIN_TRADES_TOP1).then(pl.col("top1_share")).otherwise(None).alias("top1_share"),
])

In [26]:
del bar_tail, bars_size, top1_sum, top1_share
gc.collect()

0

# Step 6: E) Sequence Proxies (Q)

prev_s = sign ก่อนหน้าใน “บาร์เดียวกัน”

sign_persistence = mean(s == prev_s) (ตัดตัวแรกที่ prev_s เป็น null ออก)

flip_count = sum(s != prev_s)

sign_entropy (sequence-based proxy) = entropy ของ transition 4 แบบ: ++ , +- , -+ , --

In [27]:
df_seq = df_bar.sort(["bar_t", "t", "aggTradeId"]).with_columns([
    pl.col("s").shift(1).over("bar_t").alias("prev_s"),
])

In [28]:
bars_seq_basic = (
    df_seq
    .group_by("bar_t")
    .agg([
        # persistence / flips (ignore null prev_s by filtering)
        pl.when(pl.col("prev_s").is_not_null())
          .then((pl.col("s") == pl.col("prev_s")).cast(pl.Float64))
          .otherwise(None)
          .mean()
          .alias("sign_persistence_inbar"),

        pl.when(pl.col("prev_s").is_not_null())
          .then((pl.col("s") != pl.col("prev_s")).cast(pl.Int64))
          .otherwise(0)
          .sum()
          .alias("flip_count"),

        # transition counts: ++, +-, -+, --
        pl.when(pl.col("prev_s").is_not_null() & (pl.col("prev_s") == 1) & (pl.col("s") == 1)).then(1).otherwise(0).sum().alias("n_pp"),
        pl.when(pl.col("prev_s").is_not_null() & (pl.col("prev_s") == 1) & (pl.col("s") == -1)).then(1).otherwise(0).sum().alias("n_pm"),
        pl.when(pl.col("prev_s").is_not_null() & (pl.col("prev_s") == -1) & (pl.col("s") == 1)).then(1).otherwise(0).sum().alias("n_mp"),
        pl.when(pl.col("prev_s").is_not_null() & (pl.col("prev_s") == -1) & (pl.col("s") == -1)).then(1).otherwise(0).sum().alias("n_mm"),
    ])
    .with_columns([
        (pl.col("n_pp") + pl.col("n_pm") + pl.col("n_mp") + pl.col("n_mm")).alias("n_trans"),
    ])
    .with_columns([
        # transition probabilities
        pl.when(pl.col("n_trans") > 0).then(pl.col("n_pp") / pl.col("n_trans")).otherwise(0.0).alias("p_pp"),
        pl.when(pl.col("n_trans") > 0).then(pl.col("n_pm") / pl.col("n_trans")).otherwise(0.0).alias("p_pm"),
        pl.when(pl.col("n_trans") > 0).then(pl.col("n_mp") / pl.col("n_trans")).otherwise(0.0).alias("p_mp"),
        pl.when(pl.col("n_trans") > 0).then(pl.col("n_mm") / pl.col("n_trans")).otherwise(0.0).alias("p_mm"),
    ])
    .with_columns([
        # entropy = -sum p log p  (0 log 0 treated as 0)
        (
            pl.sum_horizontal([
                pl.when(pl.col("p_pp") > 0).then(-pl.col("p_pp") * pl.col("p_pp").log()).otherwise(0.0),
                pl.when(pl.col("p_pm") > 0).then(-pl.col("p_pm") * pl.col("p_pm").log()).otherwise(0.0),
                pl.when(pl.col("p_mp") > 0).then(-pl.col("p_mp") * pl.col("p_mp").log()).otherwise(0.0),
                pl.when(pl.col("p_mm") > 0).then(-pl.col("p_mm") * pl.col("p_mm").log()).otherwise(0.0),
            ])
        ).alias("sign_entropy")
    ])
    .select([
        pl.col("bar_t").alias("t"),
        "sign_persistence_inbar",
        "flip_count",
        "sign_entropy",
        "n_trans",
    ])
)

In [29]:
bars_core = bars_core.join(bars_seq_basic, on="t", how="left")

In [30]:
del bars_seq_basic, df_seq
gc.collect()

0

run_max / run_mean

สร้าง run_id ภายในบาร์ด้วยการ “เริ่ม run ใหม่เมื่อ sign เปลี่ยน”

In [31]:
bars_runs = (
    df_bar
    .group_by("bar_t")
    .agg([
        pl.col("s").alias("s_list"),
    ])
    .with_columns([
        # diff sign inside list (ตัวแรกจะเป็น null)
        pl.col("s_list").list.diff().alias("ds"),
    ])
    .with_columns([
        # start of new run = ds != 0 (null ให้เป็น 1 เพื่อเริ่ม run แรก)
        pl.col("ds")
        .list.eval(
            pl.when(pl.element().is_null())
              .then(1)
              .when(pl.element() != 0)
              .then(1)
              .otherwise(0)
        )
        .alias("is_new"),
    ])
    .with_columns([
        # cumulative run id inside list  ✅ (ไม่มี list.cum_sum)
        pl.col("is_new")
        .list.eval(pl.element().cum_sum())
        .alias("run_id_list"),
    ])
    .with_columns([
        # count run lengths (นับความถี่ของ run_id)
        pl.struct(["run_id_list"])
        .map_elements(
            lambda x: list(__import__("collections").Counter(x["run_id_list"]).values())
        )
        .alias("run_lens")
    ])
    .with_columns([
        pl.col("run_lens").list.max().alias("run_max"),
        pl.col("run_lens").list.mean().alias("run_mean"),
    ])
    .select([
        pl.col("bar_t").alias("t"),
        "run_max",
        "run_mean",
    ])
)

In [32]:
bars_core = bars_core.join(bars_runs, on="t", how="left")

# Post-Process & Recompute some Feature that use prev (null) bar

In [33]:
# -------------------------
# MASK STATS WHEN SPARSE (avoid bias)
# -------------------------
bars_core = bars_core.with_columns([
    # time-geometry stats: ต้องมี trades พอ
    pl.when(pl.col("trade_count") >= MIN_TRADES_STATS).then(pl.col("dt_mean_ms")).otherwise(None).alias("dt_mean_ms"),
    pl.when(pl.col("trade_count") >= MIN_TRADES_STATS).then(pl.col("dt_median_ms")).otherwise(None).alias("dt_median_ms"),
    pl.when(pl.col("trade_count") >= MIN_TRADES_STATS).then(pl.col("dt_p90_ms")).otherwise(None).alias("dt_p90_ms"),
    pl.when(pl.col("trade_count") >= MIN_TRADES_STATS).then(pl.col("frac_dt0")).otherwise(None).alias("frac_dt0"),
    pl.when(pl.col("trade_count") >= MIN_TRADES_STATS).then(pl.col("active_ms")).otherwise(None).alias("active_ms"),
    pl.when(pl.col("trade_count") >= MIN_TRADES_STATS).then(pl.col("intensity_peak_1s")).otherwise(None).alias("intensity_peak_1s"),

    # micro price stats
    pl.when(pl.col("n_returns") >= MIN_RETURNS_RV).then(pl.col("realized_var_inbar")).otherwise(None).alias("realized_var_inbar"),
    pl.when(pl.col("n_returns") >= MIN_TRADES_STATS).then(pl.col("dp0_frac")).otherwise(None).alias("dp0_frac"),
    pl.when(pl.col("n_returns") >= MIN_TRADES_STATS).then(pl.col("n_price_changes")).otherwise(None).alias("n_price_changes"),
    pl.when(pl.col("n_returns") >= MIN_TRADES_STATS).then(pl.col("up_ticks")).otherwise(None).alias("up_ticks"),
    pl.when(pl.col("n_returns") >= MIN_TRADES_STATS).then(pl.col("down_ticks")).otherwise(None).alias("down_ticks"),
    pl.when(pl.col("n_returns") >= MIN_TRADES_STATS).then(pl.col("max_up_move")).otherwise(None).alias("max_up_move"),
    pl.when(pl.col("n_returns") >= MIN_TRADES_STATS).then(pl.col("max_down_move")).otherwise(None).alias("max_down_move"),

    # sequence stats: ต้องมี transitions พอ
    pl.when(pl.col("n_trans") >= MIN_TRADES_ENTROPY).then(pl.col("sign_persistence_inbar")).otherwise(None).alias("sign_persistence_inbar"),
    pl.when(pl.col("n_trans") >= MIN_TRADES_ENTROPY).then(pl.col("flip_count")).otherwise(None).alias("flip_count"),
    pl.when(pl.col("n_trans") >= MIN_TRADES_ENTROPY).then(pl.col("sign_entropy")).otherwise(None).alias("sign_entropy"),
])

In [34]:
bars_core = bars_core.with_columns([
    pl.when(pl.col("n_trans") >= MIN_TRADES_ENTROPY).then(pl.col("run_max")).otherwise(None).alias("run_max"),
    pl.when(pl.col("n_trans") >= MIN_TRADES_ENTROPY).then(pl.col("run_mean")).otherwise(None).alias("run_mean"),
])

In [35]:
bars_core = bars_core.with_columns([
    pl.when(pl.col("trade_count") >= MIN_TRADES_STATS).then(pl.col("imbalance")).otherwise(None).alias("imbalance"),
])

In [36]:
bars_core

t,open,high,low,close,vol_base,vol_quote,trade_count,buy_count,sell_count,buy_quote,sell_quote,signed_quote,vwap,trade_rate,imbalance,range,hl_pct,prev_close,ret_inbar,ret_cc,gap_oc,dt_mean_ms,dt_median_ms,dt_p90_ms,frac_dt0,active_ms,intensity_peak_1s,realized_var_inbar,dp0_frac,n_price_changes,up_ticks,down_ticks,max_up_move,max_down_move,n_returns,avg_trade_q,avg_trade_notional,q_p50,q_p90,q_p99,notional_p90,notional_p99,top1_share,sign_persistence_inbar,flip_count,sign_entropy,n_trans,run_max,run_mean
datetime[ms],f64,f64,f64,f64,f64,f64,u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,u32,f64,f64,u32,u32,u32,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i32,i64,f64
2026-01-01 00:00:00,4337.0,4337.0,4336.99,4337.0,0.0232,100.618365,6,5,1,85.4389,15.179465,70.259435,4336.998491,0.1,0.698276,0.01,0.000002,null,0.0,null,null,8980.2,10575.0,18781.0,0.0,6,2,1.0633e-11,0.6,2,1,1,0.01,-0.01,5,0.003867,16.769727,0.0035,0.007,0.007,30.359,30.359,null,null,null,null,5,null,null
2026-01-01 00:01:00,4337.0,4339.45,4336.99,4339.44,15.6357,67822.971711,44,39,5,66154.560186,1668.411525,64486.148661,4337.699733,0.733333,0.950801,2.46,0.000567,4337.0,0.000562,0.000562,0.0,1301.465116,0.0,4148.0,0.627907,17,34,2.1783e-8,0.116279,38,34,4,0.39,-0.01,43,0.355357,1541.431175,0.103,1.0984,2.2238,4764.20016,9644.6206,null,0.790698,9,0.656813,43,28,4.4
2026-01-01 00:02:00,4339.45,4339.45,4339.45,4339.45,0.0424,183.99268,6,6,0,183.99268,0.0,183.99268,4339.45,0.1,1.0,0.0,0.0,4339.44,0.0,0.000002,0.000002,7312.2,3976.0,17088.0,0.2,5,3,0.0,1.0,0,0,0,0.0,0.0,5,0.007067,30.665447,0.007,0.0075,0.0075,32.545875,32.545875,null,null,null,null,5,null,null
2026-01-01 00:03:00,4339.45,4339.67,4339.45,4339.67,6.3133,27396.316873,8,8,0,27396.316873,0.0,27396.316873,4339.460642,0.133333,1.0,0.22,0.000051,4339.45,0.000051,0.000051,0.0,1502.428571,16.0,4543.0,0.0,8,6,1.2851e-9,0.714286,2,2,0,0.11,0.0,7,0.789162,3424.539609,0.007,0.2765,6.0,1199.918755,26036.7,null,null,null,null,7,null,null
2026-01-01 00:04:00,4339.67,4340.76,4339.67,4340.76,5.7104,24783.365706,24,23,1,24729.540406,53.8253,24675.715106,4340.040226,0.4,0.995656,1.09,0.000251,4339.67,0.000251,0.000251,0.0,1928.869565,23.0,5519.0,0.130435,21,10,8.2122e-9,0.391304,14,13,1,0.21,-0.01,23,0.237933,1032.640238,0.0105,0.5594,3.0946,2428.042136,13429.542782,null,0.913043,2,0.355713,23,22,8.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-01-28 23:55:00,5515.96,5526.39,5515.43,5524.8,62.7264,346144.720537,371,214,157,205229.250795,140915.469742,64313.781053,5518.325945,6.183333,0.1858,10.96,0.001986,5515.96,0.001601,0.001601,0.0,158.221622,1.0,512.0,0.481081,193,43,4.8031e-7,0.232432,284,191,93,0.64,-1.01,370,0.169074,933.004638,0.0247,0.4,1.6515,2206.172,9115.02486,0.232932,0.808108,71,1.167308,370,45,5.152778
2026-01-28 23:56:00,5524.61,5553.28,5524.61,5548.43,92.6212,513290.627601,702,450,252,314735.118608,198555.508993,116179.609615,5541.826575,11.7,0.226343,28.67,0.005173,5524.8,0.004302,0.004268,-0.000034,84.045649,0.0,277.0,0.586305,291,44,0.000001,0.135521,606,402,204,1.22,-0.8,701,0.131939,731.18323,0.0056,0.4,1.725,2217.132,9538.905,0.198774,0.908702,64,0.954502,701,63,10.8
2026-01-28 23:57:00,5548.8,5553.88,5545.08,5549.95,80.657,447609.51093,481,212,269,213657.478777,233952.032153,-20294.553376,5549.543263,8.016667,-0.04534,8.8,0.001586,5548.43,0.000207,0.000274,0.000067,121.402083,0.0,408.0,0.545833,219,66,6.5064e-7,0.233333,368,182,186,0.96,-0.78,480,0.167686,930.581104,0.0381,0.5047,1.4523,2799.379114,8065.899924,0.147904,0.895833,50,1.019671,480,66,9.431373


In [37]:
# =========================
# FIX: build complete bar grid + fill no-trade + label missing
# =========================

gap_threshold_ms = 10 * 60 * 1000  # 10 นาที

# 0) ทำให้ bars_core เรียงเวลาแน่นอน
bars_core = bars_core.sort("t")

# 1) สร้าง time grid ครบทุกบาร์ (bar_seconds)
t_min = df0.select(pl.col("t").min()).item()
t_max = df0.select(pl.col("t").max()).item()

# align start/end ให้ตรง boundary ของ bar
start = pl.Series([t_min]).dt.truncate(f"{bar_seconds}s").item()
end   = pl.Series([t_max]).dt.truncate(f"{bar_seconds}s").item()

grid = pl.DataFrame({
    "t": pl.datetime_range(
        start=start,
        end=end + pl.duration(seconds=bar_seconds),   # ให้ include บาร์สุดท้าย
        interval=f"{bar_seconds}s",
        closed="left",
        eager=True,
    ).cast(pl.Datetime("ms"))
})

# 2) left join grid กับ bars_core -> ได้บาร์ที่ "ไม่มี trade" เป็น null
bars = (
    grid
    .join(bars_core, on="t", how="left")
    .with_columns([
        (pl.col("trade_count").fill_null(0) > 0).alias("has_trade"),
    ])
)

bars = bars.with_columns([
    (pl.col("trade_count").fill_null(0) < MIN_TRADES_STATS).alias("is_sparse_stats"),
    (pl.col("trade_count").fill_null(0) < MIN_TRADES_ENTROPY).alias("is_sparse_seq"),
    (pl.col("trade_count").fill_null(0) < MIN_TRADES_TOP1).alias("is_sparse_top1"),
])

# 3) หา "เวลาบาร์ล่าสุดที่มี trade" เพื่อวัด gap (ใช้ forward_fill ของ t เฉพาะแถวที่มี trade)
bars = (
    bars
    .with_columns([
        pl.when(pl.col("has_trade"))
          .then(pl.col("t"))
          .otherwise(None)
          .alias("_t_trade")
    ])
    .with_columns([
        pl.col("_t_trade").fill_null(strategy="forward").alias("last_trade_t"),
    ])
    .with_columns([
        (pl.col("t") - pl.col("last_trade_t")).dt.total_milliseconds().alias("gap_ms_from_last_trade"),
    ])
)

# 4) label: no-trade-fill (gap <= 10m) vs missing-fill (gap > 10m)
bars = (
    bars
    .with_columns([
        # แถวที่ไม่มี trade จริง
        (~pl.col("has_trade")).alias("is_empty_bar"),
    ])
    .with_columns([
        pl.when(pl.col("is_empty_bar") & pl.col("last_trade_t").is_not_null() &
                (pl.col("gap_ms_from_last_trade") <= gap_threshold_ms))
          .then(True).otherwise(False).alias("is_no_trade_fill"),

        pl.when(pl.col("is_empty_bar") &
                (pl.col("last_trade_t").is_null() | (pl.col("gap_ms_from_last_trade") > gap_threshold_ms)))
          .then(True).otherwise(False).alias("is_missing_fill"),
    ])
)

# 5) เติมราคาให้เฉพาะ no-trade-fill (ใช้ราคาล่าสุดก่อนหน้า)
# ทำ carried price ก่อน (forward fill) แล้ว mask ด้วย is_no_trade_fill
bars = (
    bars
    .with_columns([
        # forward-filled close สำหรับใช้เป็นราคาอ้างอิง
        pl.col("close").fill_null(strategy="forward").alias("_close_ffill"),
        pl.col("vwap").fill_null(strategy="forward").alias("_vwap_ffill"),
    ])
    .with_columns([
        # สำหรับบาร์ว่างแบบ no-trade: OHLC=vwap=last close
        pl.when(pl.col("is_no_trade_fill")).then(pl.col("_close_ffill")).otherwise(pl.col("open")).alias("open"),
        pl.when(pl.col("is_no_trade_fill")).then(pl.col("_close_ffill")).otherwise(pl.col("high")).alias("high"),
        pl.when(pl.col("is_no_trade_fill")).then(pl.col("_close_ffill")).otherwise(pl.col("low")).alias("low"),
        pl.when(pl.col("is_no_trade_fill")).then(pl.col("_close_ffill")).otherwise(pl.col("close")).alias("close"),
        pl.when(pl.col("is_no_trade_fill")).then(pl.col("_close_ffill")).otherwise(pl.col("vwap")).alias("vwap"),
    ])
    .drop(["_close_ffill", "_vwap_ffill"])
)

# 6) เติมค่า "facts" สำหรับ no-trade-fill ให้เป็น 0/ค่าเหมาะสม
#    แต่สำหรับ missing_fill ให้ปล่อยเป็น null (เพื่อไม่หลอกโมเดล)
zero_fill_facts = [
    # activity / flow
    "trade_count", "trade_rate",
    "buy_count", "sell_count",
    "buy_quote", "sell_quote", "signed_quote",

    # volume / size
    "vol_base", "vol_quote",
    "avg_trade_q", "avg_trade_notional",

    # (optional) quantiles ของ q/notional: ถ้า no-trade-fill ให้ 0 ก็โอเค เพราะไม่มี trade จริง
    "q_p50", "q_p90", "q_p99",
    "notional_p90", "notional_p99",
]

# เติม 0 เฉพาะ no-trade-fill (กันคอลัมน์ที่ไม่มีใน df)
existing_zero_cols = [c for c in zero_fill_facts if c in bars.columns]

bars = bars.with_columns([
    pl.when(pl.col("is_no_trade_fill"))
      .then(pl.lit(0.0))
      .otherwise(pl.col(c))
      .alias(c)
    for c in existing_zero_cols
])

# 7) recompute derived fields ที่พึ่งพา price หลัง fill (range/hl_pct/ret/ret_cc/gap_oc/imbalance ฯลฯ)
bars = (
    bars
    .with_columns([
        (pl.col("high") - pl.col("low")).alias("range"),
        pl.when(pl.col("vwap") > 0).then((pl.col("high") - pl.col("low")) / pl.col("vwap")).otherwise(None).alias("hl_pct"),
    ])
    .with_columns([
        # prev_close ใหม่หลัง fill (บน time grid)
        pl.col("close").shift(1).alias("prev_close"),
    ])
    .with_columns([
        pl.when((pl.col("open") > 0) & (pl.col("close") > 0))
          .then((pl.col("close") / pl.col("open")).log())
          .otherwise(None)
          .alias("ret_inbar"),

        pl.when((pl.col("prev_close") > 0) & (pl.col("close") > 0))
          .then((pl.col("close") / pl.col("prev_close")).log())
          .otherwise(None)
          .alias("ret_cc"),

        pl.when((pl.col("prev_close") > 0) & (pl.col("open") > 0))
          .then((pl.col("open") / pl.col("prev_close")).log())
          .otherwise(None)
          .alias("gap_oc"),

        pl.when(pl.col("vol_quote") > 0)
          .then(pl.col("signed_quote") / pl.col("vol_quote"))
          .otherwise(None)
          .alias("imbalance"),
    ])
)

# 8) ถ้าคุณอยาก "ห้าม forward-fill ราคาใน missing_fill" แบบเข้ม:
#    -> ทำให้ OHLC/VWAP เป็น null ในช่วง missing_fill (คงไว้เฉพาะ label)
bars = bars.with_columns([
    pl.when(pl.col("is_missing_fill")).then(None).otherwise(pl.col("open")).alias("open"),
    pl.when(pl.col("is_missing_fill")).then(None).otherwise(pl.col("high")).alias("high"),
    pl.when(pl.col("is_missing_fill")).then(None).otherwise(pl.col("low")).alias("low"),
    pl.when(pl.col("is_missing_fill")).then(None).otherwise(pl.col("close")).alias("close"),
    pl.when(pl.col("is_missing_fill")).then(None).otherwise(pl.col("vwap")).alias("vwap"),
])

# ผลลัพธ์สุดท้ายคือ bars (time-grid ครบ + มี flags)
# bars: มีคอลัมน์ is_no_trade_fill / is_missing_fill ให้คุณใช้ downstream ได้เลย

# AggBars

In [38]:
bars = bars.drop([
    "has_trade",
    "_t_trade",
    "last_trade_t",
    "is_empty_bar"
])

In [39]:
bars

t,open,high,low,close,vol_base,vol_quote,trade_count,buy_count,sell_count,buy_quote,sell_quote,signed_quote,vwap,trade_rate,imbalance,range,hl_pct,prev_close,ret_inbar,ret_cc,gap_oc,dt_mean_ms,dt_median_ms,dt_p90_ms,frac_dt0,active_ms,intensity_peak_1s,realized_var_inbar,dp0_frac,n_price_changes,up_ticks,down_ticks,max_up_move,max_down_move,n_returns,avg_trade_q,avg_trade_notional,q_p50,q_p90,q_p99,notional_p90,notional_p99,top1_share,sign_persistence_inbar,flip_count,sign_entropy,n_trans,run_max,run_mean,is_sparse_stats,is_sparse_seq,is_sparse_top1,gap_ms_from_last_trade,is_no_trade_fill,is_missing_fill
datetime[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,u32,f64,f64,u32,u32,u32,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,i32,i64,f64,bool,bool,bool,i64,bool,bool
2026-01-01 00:00:00,4337.0,4337.0,4336.99,4337.0,0.0232,100.618365,6.0,5.0,1.0,85.4389,15.179465,70.259435,4336.998491,0.1,0.698276,0.01,0.000002,null,0.0,null,null,8980.2,10575.0,18781.0,0.0,6,2,1.0633e-11,0.6,2,1,1,0.01,-0.01,5,0.003867,16.769727,0.0035,0.007,0.007,30.359,30.359,null,null,null,null,5,null,null,false,true,true,0,false,false
2026-01-01 00:01:00,4337.0,4339.45,4336.99,4339.44,15.6357,67822.971711,44.0,39.0,5.0,66154.560186,1668.411525,64486.148661,4337.699733,0.733333,0.950801,2.46,0.000567,4337.0,0.000562,0.000562,0.0,1301.465116,0.0,4148.0,0.627907,17,34,2.1783e-8,0.116279,38,34,4,0.39,-0.01,43,0.355357,1541.431175,0.103,1.0984,2.2238,4764.20016,9644.6206,null,0.790698,9,0.656813,43,28,4.4,false,false,true,0,false,false
2026-01-01 00:02:00,4339.45,4339.45,4339.45,4339.45,0.0424,183.99268,6.0,6.0,0.0,183.99268,0.0,183.99268,4339.45,0.1,1.0,0.0,0.0,4339.44,0.0,0.000002,0.000002,7312.2,3976.0,17088.0,0.2,5,3,0.0,1.0,0,0,0,0.0,0.0,5,0.007067,30.665447,0.007,0.0075,0.0075,32.545875,32.545875,null,null,null,null,5,null,null,false,true,true,0,false,false
2026-01-01 00:03:00,4339.45,4339.67,4339.45,4339.67,6.3133,27396.316873,8.0,8.0,0.0,27396.316873,0.0,27396.316873,4339.460642,0.133333,1.0,0.22,0.000051,4339.45,0.000051,0.000051,0.0,1502.428571,16.0,4543.0,0.0,8,6,1.2851e-9,0.714286,2,2,0,0.11,0.0,7,0.789162,3424.539609,0.007,0.2765,6.0,1199.918755,26036.7,null,null,null,null,7,null,null,false,true,true,0,false,false
2026-01-01 00:04:00,4339.67,4340.76,4339.67,4340.76,5.7104,24783.365706,24.0,23.0,1.0,24729.540406,53.8253,24675.715106,4340.040226,0.4,0.995656,1.09,0.000251,4339.67,0.000251,0.000251,0.0,1928.869565,23.0,5519.0,0.130435,21,10,8.2122e-9,0.391304,14,13,1,0.21,-0.01,23,0.237933,1032.640238,0.0105,0.5594,3.0946,2428.042136,13429.542782,null,0.913043,2,0.355713,23,22,8.0,false,false,true,0,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-01-28 23:55:00,5515.96,5526.39,5515.43,5524.8,62.7264,346144.720537,371.0,214.0,157.0,205229.250795,140915.469742,64313.781053,5518.325945,6.183333,0.1858,10.96,0.001986,5515.96,0.001601,0.001601,0.0,158.221622,1.0,512.0,0.481081,193,43,4.8031e-7,0.232432,284,191,93,0.64,-1.01,370,0.169074,933.004638,0.0247,0.4,1.6515,2206.172,9115.02486,0.232932,0.808108,71,1.167308,370,45,5.152778,false,false,false,0,false,false
2026-01-28 23:56:00,5524.61,5553.28,5524.61,5548.43,92.6212,513290.627601,702.0,450.0,252.0,314735.118608,198555.508993,116179.609615,5541.826575,11.7,0.226343,28.67,0.005173,5524.8,0.004302,0.004268,-0.000034,84.045649,0.0,277.0,0.586305,291,44,0.000001,0.135521,606,402,204,1.22,-0.8,701,0.131939,731.18323,0.0056,0.4,1.725,2217.132,9538.905,0.198774,0.908702,64,0.954502,701,63,10.8,false,false,false,0,false,false
2026-01-28 23:57:00,5548.8,5553.88,5545.08,5549.95,80.657,447609.51093,481.0,212.0,269.0,213657.478777,233952.032153,-20294.553376,5549.543263,8.016667,-0.04534,8.8,0.001586,5548.43,0.000207,0.000274,0.000067,121.402083,0.0,408.0,0.545833,219,66,6.5064e-7,0.233333,368,182,186,0.96,-0.78,480,0.167686,930.581104,0.0381,0.5047,1.4523,2799.3791

In [40]:
bars.write_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/PAXGUSDTaggTrades/aggBars-1m-PAXG-2026-01-28.parquet")))

# why time warp

In [ ]:
# bars_core = pl.read_parquet("/content/drive/MyDrive/LoserToWinner/0.Data -> Facts/Binance Vision Data/aggBars/aggBars-5s-ETH-2018.parquet")

In [ ]:
# import polars as pl
# df = pl.read_parquet("/content/drive/MyDrive/LoserToWinner/0.Data -> Facts/Binance Vision Data/aggTrades/aggTrades-ETH-2020.parquet")

In [ ]:
# df

In [ ]:
# gaps = (
#     df.select("timestamp").sort("timestamp")
#     .with_columns([
#         pl.col("timestamp").shift(1).alias("t_prev"),
#         (pl.col("timestamp") - pl.col("timestamp").shift(1)).alias("dt"),
#     ])
#     .filter(pl.col("dt") > pl.duration(minutes=1))
# )

# gaps.sort("dt", descending=True)

In [ ]:
# gaps.sort("dt", descending=True)[:10]

In [ ]:
# import requests
# from pathlib import Path

# url = "https://data.binance.vision/data/spot/monthly/aggTrades/ETHUSDT/ETHUSDT-aggTrades-2018-02.zip"
# out_path = Path("ETHUSDT-aggTrades-2018-02.zip")

# r = requests.get(url, stream=True)
# r.raise_for_status()

# with open(out_path, "wb") as f:
#     for chunk in r.iter_content(chunk_size=1024 * 1024):
#         f.write(chunk)

# out_path

In [ ]:
# import zipfile

# with zipfile.ZipFile(out_path, "r") as z:
#     z.extractall("ETHUSDT-aggTrades-2018-02")

# Path("ETHUSDT-aggTrades-2018-02").iterdir()

In [ ]:
# import polars as pl

# cols = [
#     "aggTradeId",
#     "price",
#     "quantity",
#     "firstTradeId",
#     "lastTradeId",
#     "timestamp",
#     "isBuyerMaker",
#     "isBestMatch",
# ]

# df = pl.read_csv(
#     "ETHUSDT-aggTrades-2018-02/ETHUSDT-aggTrades-2018-02.csv",
#     has_header=False,
#     new_columns=cols,
# )

In [ ]:
# df = df.with_columns(
#     pl.col("timestamp").cast(pl.Datetime("ms"))
# )

In [ ]:
# df_t = df.select("timestamp").sort("timestamp")

# df_t.with_columns(
#     (pl.col("timestamp") < pl.col("timestamp").shift(1)).alias("is_decreasing")
# ).select(
#     pl.col("is_decreasing").sum().alias("n_decreasing")
# )

In [ ]:
# df_sorted = df.select(["timestamp", "aggTradeId"]).sort(["timestamp", "aggTradeId"])

# gaps = (
#     df_sorted
#     .with_columns(
#         pl.col("timestamp").shift(1).alias("t_prev"),
#         (pl.col("timestamp") - pl.col("timestamp").shift(1)).alias("dt"),
#     )
#     .filter(pl.col("dt") > pl.duration(minutes=1))
# )

# gaps.sort("dt", descending=True)